In [ ]:
import pandas as pd
import sys
import os
from sklearn import set_config

from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import train_test_split
from typing import Tuple
set_config(display="text")  # displays text representation of estimators

from sksurv.util import Surv
sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
from src.utils.cox_models import Cox_regression, p_values_Cox_regression
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
pp = Preprocessor()
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_keep = pp.clean_columns_dataset(df_clinical_data)



In [3]:
list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df
df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
df_rsem_z_scores = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt")
clean_df = pp.eliminate_nan_genes(df_rsem_z_scores, "Hugo_Symbol")



Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 298 - Total(%) 0.36
Shape of the CSV: (20440, 819)
Shape of the CSV: (20440, 819)
Max NaN per gene: 817
Genes before: 20440
Genes after: 20107


In [4]:

"""expression = df_ESR1_merged["expression"].values
lsit = sorted(expression)
plt.plot((lsit)) # plotting by columns
plt.show()
"""

'expression = df_ESR1_merged["expression"].values\nlsit = sorted(expression)\nplt.plot((lsit)) # plotting by columns\nplt.show()\n'

In [5]:

"""
ESR1 = df_ESR1_merged["expression"].values.reshape(-1,1)

scaler = StandardScaler()
z_score = scaler.fit_transform(ESR1)

plt.figure(figsize=(12,5))
plt.title("box plot of the z score")
plt.boxplot(z_score)
plt.show()

plt.figure(figsize=(12,5))
plt.hist(z_score, bins=30)
plt.title("Distribution of the z score")
plt.xlabel("z-score")
plt.ylabel("Frecuencia")
plt.show()
print(f"Min number ESR1: {ESR1.min()}")
print(f"Max number ESR1: {ESR1.max()}")
#plt.xlabel(f"Max number in ESR1:{ESR1.max()} ")"""

'\nESR1 = df_ESR1_merged["expression"].values.reshape(-1,1)\n\nscaler = StandardScaler()\nz_score = scaler.fit_transform(ESR1)\n\nplt.figure(figsize=(12,5))\nplt.title("box plot of the z score")\nplt.boxplot(z_score)\nplt.show()\n\nplt.figure(figsize=(12,5))\nplt.hist(z_score, bins=30)\nplt.title("Distribution of the z score")\nplt.xlabel("z-score")\nplt.ylabel("Frecuencia")\nplt.show()\nprint(f"Min number ESR1: {ESR1.min()}")\nprint(f"Max number ESR1: {ESR1.max()}")\n#plt.xlabel(f"Max number in ESR1:{ESR1.max()} ")'

In [6]:
def split_data_set(df_mRNA : pd.DataFrame, 
                       df_clinical : pd.DataFrame,
                       gene : str) -> Tuple:
    
        df_single_gene  = pp.gene_to_long(df_mRNA, gene)
        df_gene_merged = df_single_gene.merge(df_clinical, on="Sample ID", how="inner")
        
        status = df_gene_merged["Overall Survival Status"].astype(str).str.strip()
        
        df_gene_merged["event"] = status.str.contains("DECEASED", na=False) 
    
        df_gene_merged = df_gene_merged.dropna(subset=["Overall Survival (Months)"])
        
        df_gene_merged["expression"] = pd.to_numeric(df_gene_merged["expression"], errors="coerce")
        
        X = df_gene_merged[["expression"]]
        
        Y_surv = Surv.from_dataframe(
        event="event",
        time="Overall Survival (Months)",
        data=df_gene_merged
        )
        
        return X, Y_surv, df_gene_merged

In [7]:
X_MKI67, Y_surv_MKI67, df_gene_merged_MKI67 = split_data_set(clean_df,df_clinical_keep, "MKI67")
X_train_MKI67, X_test_MKI67, Y_train_MKI67, Y_test_MKI67 = train_test_split(
    X_MKI67, Y_surv_MKI67, train_size=0.80, test_size=0.20, random_state=42
)
betas_MKI67, chp_predict_MKI673, survival_curve_MKI67, risk_curve_MKI67 = Cox_regression(X_train_MKI67, Y_train_MKI67, X_test_MKI67)

df_life_MKI67 = df_gene_merged_MKI67[["expression", "event", "Overall Survival (Months)"]]
df_life_MKI67 = df_life_MKI67.dropna(subset="Overall Survival (Months)")
df_life_MKI67 = df_life_MKI67.dropna(subset="expression")
p_values_Cox_regression(df_life_MKI67,event_col="event", duration_col="Overall Survival (Months)")

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
expression,0.085822,1.089613,0.089302,-0.089206,0.260851,0.914657,1.298034,0.0,0.961037,0.336534,1.571178


In [8]:
genes_expression = [
    "TBC1D9",
    "SUSD3",
    "SLC39A6",
    "GFRA1",
    "SOX11",
    "GATA3",
    "SLC15A2",
    "NANOS1",
    "ZNF552",
    "ESR1",
    "NAT1",
    "NME3",
    "DNALI1",
    "AGR3",
    "CA12",
    "BCL2",
    "MKI67",
    "TP53",
    "ESR1",
    "AURKA"
]
genes_significant_p_value = []
genes_non_significant_p_value = []

for gene_sample in genes_expression:
    X, Y_surv, df_gene_merged = split_data_set(clean_df,df_clinical_keep, gene_sample)
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y_surv, train_size=0.80, test_size=0.20
    )
    beta, chp_predict, survival_curv, risk_curve = Cox_regression(X_train, Y_train, X_test)

    df_life = df_gene_merged[["expression", "event", "Overall Survival (Months)"]]
    df_life = df_life.dropna(subset="Overall Survival (Months)")
    df_life = df_life.dropna(subset="expression")
    value = p_values_Cox_regression(df_life,event_col="event", duration_col="Overall Survival (Months)")
    p = value["p"].item()
    if p <= 0.05:
        genes_significant_p_value.append(f"Gene: {gene_sample} and the P-value: {p}")
    else:
        genes_non_significant_p_value.append(f"Gene: {gene_sample} and the P-value: {p}")

In [9]:
genes_significant_p_value

['Gene: SUSD3 and the P-value: 0.0010301670185654543',
 'Gene: BCL2 and the P-value: 0.01726417415138832']

In [10]:
genes_non_significant_p_value

['Gene: TBC1D9 and the P-value: 0.6819163660067763',
 'Gene: SLC39A6 and the P-value: 0.19976711022845564',
 'Gene: GFRA1 and the P-value: 0.1258221247638053',
 'Gene: SOX11 and the P-value: 0.16650002142967513',
 'Gene: GATA3 and the P-value: 0.21529888861308868',
 'Gene: SLC15A2 and the P-value: 0.5246819853527852',
 'Gene: NANOS1 and the P-value: 0.0802611504345009',
 'Gene: ZNF552 and the P-value: 0.1648315796274484',
 'Gene: ESR1 and the P-value: 0.9547188662523933',
 'Gene: NAT1 and the P-value: 0.6701425922125758',
 'Gene: NME3 and the P-value: 0.19542251342268882',
 'Gene: DNALI1 and the P-value: 0.33723756440052244',
 'Gene: AGR3 and the P-value: 0.4620363491487117',
 'Gene: CA12 and the P-value: 0.511559098069994',
 'Gene: MKI67 and the P-value: 0.33653360502829066',
 'Gene: TP53 and the P-value: 0.1138474750512458',
 'Gene: ESR1 and the P-value: 0.9547188662523933',
 'Gene: AURKA and the P-value: 0.3012993605247465']